In [6]:
import json
import numpy as np
import torch

# Create alignment for sphere

In [7]:
def compute_sphere_low_dim_errors(path):
    from scipy.spatial import cKDTree

    def spherical_to_cartesian(theta, phi):
        x = torch.sin(theta) * torch.cos(phi)
        y = torch.sin(theta) * torch.sin(phi)
        z = torch.cos(theta)
        return torch.stack([x, y, z], dim=1)

    def compute_rotation_matrix(source, target, weights):
        W = torch.diag(weights)
        A = source.T @ W @ target
        U, _, Vt = torch.svd(A)
        R = Vt @ U.T
        if torch.det(R) < 0:
            Vt[-1, :] *= -1
            R = Vt @ U.T
        return R

    def mse(a, b):
        return torch.mean((a - b) ** 2).item()

    def smape(a, b, eps=1e-8):
        return 100 * torch.mean(2 * torch.abs(a - b) / (torch.abs(a) + torch.abs(b) + eps)).item()

    # Load data
    data = torch.load(path, map_location='cpu', weights_only=False)
    latents = data["points_sub"][1].float()
    label_angles = data["curvatures_sub"][0].float()
    curvature_weights = data["curvatures_sub"][6].float()
    ground_truth = data["curvatures_sub"][5].float()

    # Build spherical grid
    n = int(np.sqrt(len(curvature_weights)))
    eps = 1e-4
    thetas = torch.linspace(0.01, torch.pi - eps, n)
    phis = torch.linspace(eps, 2 * torch.pi - eps, n)
    grid_angles = torch.cartesian_prod(thetas, phis)
    vecs = spherical_to_cartesian(grid_angles[:, 0], grid_angles[:, 1])

    # Compute rotation
    label_cartesian = spherical_to_cartesian(label_angles[:, 0], label_angles[:, 1])
    R = compute_rotation_matrix(latents, label_cartesian, curvature_weights)

    # Rotate function
    vecs_rotated = vecs @ R
    tree = cKDTree(vecs.numpy())
    _, nn_indices = tree.query(vecs_rotated.numpy(), k=1)
    rotated_curvature = curvature_weights[nn_indices]

    return mse(rotated_curvature, ground_truth), smape(rotated_curvature, ground_truth)


# Generate Tables

# VAE Complete Table

In [8]:
def format_value(val, unit="", highlight=None):
    if val == "-":
        return "-"
    v = f"{val:.2f}\\%" if unit == "%" else f"{val:.4f}"
    if highlight == "bold":
        return r"\textbf{" + v + "}"
    elif highlight == "underline":
        return r"\underline{" + v + "}"
    return v


def generate_curv_table_from_json_lookup(file_lookup, datasets, save_path):
    betas = [0, 0.08, 1]
    gammas = [0, 1, 100]
    dtopos = [0, 1, 2]

    # Step 1: collect full grid of data
    all_rows = []  # stores row-wise [meta, mse_values, smape_values]
    if "Sphere low dim" in file_lookup:
        canonical_order = list(file_lookup["Sphere low dim"].keys())
        print("cannonical order",canonical_order)
    else:
        canonical_order = [
            (beta, gamma, d)
            for beta in betas
            for gamma in gammas
            for d in dtopos
            if not (gamma == 0 and beta != 0) and not (gamma == 0 and beta == 0 and d != 0)
        ]
    for (beta, gamma, d) in canonical_order:
        meta = (f"{beta}", f"{gamma}", f"{d}")
        mse_values = []
        smape_values = []

        for ds in datasets:
            path = file_lookup.get(ds, {}).get((beta, gamma, d), None)
            if path is None:
                mse_values.append("-")
                smape_values.append("-")
                continue
            try:
                if path.endswith(".pt"):
                    try:
                        mse, smape = compute_sphere_low_dim_errors(path)
                        print(mse,smape)
                        mse_values.append(mse)
                        smape_values.append(smape)
                    except Exception as e:
                        print(f"Error processing sphere .pt file {path}: {e}")
                        mse_values.append("-")
                        smape_values.append("-")
                elif path.endswith(".json"):
                    with open(path, "r") as f:
                        json_data = json.load(f)
                    mse = json_data["errors"]["MSE"][6]
                    smape = json_data["errors"]["SMAPE_percent"][6]
                    mse_values.append(mse)
                    smape_values.append(smape)
                else:
                    raise NotImplementedError
            except Exception as e:
                print(f"Error reading {path}: {e}")
                mse_values.append("-")
                smape_values.append("-")

        all_rows.append((meta, mse_values, smape_values))

    # Step 2: compute column-wise highlights
    n_datasets = len(datasets)
    mse_column = [[] for _ in range(n_datasets)]
    smape_column = [[] for _ in range(n_datasets)]

    for row_id, (_, mse_vals, smape_vals) in enumerate(all_rows):
        for i in range(n_datasets):
            if mse_vals[i] != "-":
                mse_column[i].append((row_id, mse_vals[i]))
            if smape_vals[i] != "-":
                smape_column[i].append((row_id, smape_vals[i]))

    mse_highlight = dict()
    smape_highlight = dict()

    for i in range(n_datasets):
        if len(mse_column[i]) > 0:
            sorted_mse = sorted(mse_column[i], key=lambda x: x[1])
            mse_highlight[(sorted_mse[0][0], i)] = "bold"
            if len(sorted_mse) > 1:
                mse_highlight[(sorted_mse[1][0], i)] = "underline"
        if len(smape_column[i]) > 0:
            sorted_smape = sorted(smape_column[i], key=lambda x: x[1])
            smape_highlight[(sorted_smape[0][0], i)] = "bold"
            if len(sorted_smape) > 1:
                smape_highlight[(sorted_smape[1][0], i)] = "underline"

    # Step 3: render table
    lines = []
    lines.append(r"\begin{table}[ht]")
    lines.append(r"\scriptsize")
    lines.append(r"    \centering")
    lines.append(f"    \\begin{{tabular}}{{lll|{'cc|' * n_datasets}}}")
    lines.append(r"        \toprule")
    header = "        $\\beta$ & $\\gamma$ & $d_{\\text{topo}}$"
    for name in datasets:
        header += f" & \\multicolumn{{2}}{{c|}}{{{name}}}"
    lines.append(header + r" \\")
    lines.append("        & & & " + " & ".join(["MSE & SMAPE"] * n_datasets) + r" \\")
    lines.append(r"        \midrule")

    for row_id, (meta, mse_vals, smape_vals) in enumerate(all_rows):
        print("meta",meta)
        alpha, gamma, dtopo = meta
        dtopo_display = "-" if gamma == "0" else dtopo
        row = [alpha, gamma, dtopo_display]
        for i in range(n_datasets):
            mse_fmt = format_value(mse_vals[i], highlight=mse_highlight.get((row_id, i)))
            smape_fmt = format_value(smape_vals[i], unit="%", highlight=smape_highlight.get((row_id, i)))
            row.append(mse_fmt)
            row.append(smape_fmt)
        lines.append("        " + " & ".join(row) + r" \\")

    lines.append(r"        \bottomrule")
    lines.append(r"    \end{tabular}")
    lines.append(r"    \caption{Curvature metrics across models and datasets.}")
    lines.append(r"    \label{tab:curv_full}")
    lines.append(r"\end{table}")

    with open(save_path, "w") as f:
        f.write("\n".join(lines))

    print(f"LaTeX table written to: {save_path}")


In [10]:
base_path_flower = "notebooks_m_vae/results/s1_synthetic/"
base_path_scrunchy = "notebooks_m_vae/results/scrunchy_dim_n/"
base_path = "notebooks_m_vae/curvatures/"

file_lookup = {
    "Curve low dim": {
        (0, 0, 0): base_path_flower + "results_exp00_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0.08, 0, 0): base_path_flower + "results_exp01_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (1, 0, 0): base_path_flower + "results_exp02_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0, 1, 0): base_path_flower + "results_exp03_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0.08, 1, 0): base_path_flower + "results_exp04_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (1, 1, 0): base_path_flower + "results_exp05_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0, 100, 0): base_path_flower + "results_exp06_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0.08, 100, 0): base_path_flower + "results_exp07_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (1, 100, 0): base_path_flower + "results_exp08_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0, 1, 1): base_path_flower + "results_exp09_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0.08, 1, 1): base_path_flower + "results_exp10_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (1, 1, 1): base_path_flower + "results_exp11_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0, 100, 1): base_path_flower + "results_exp12_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (0.08, 100, 1): base_path_flower + "results_exp13_flower_topo_VAE_100epochs/curvature_errors_stats.json",
        (1, 100, 1): base_path_flower + "results_exp14_flower_topo_VAE_100epochs/curvature_errors_stats.json",
    },
    "Curve high dim": {
        (0, 0, 0): base_path_scrunchy + "results_exp00_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0.08, 0, 0): base_path_scrunchy + "results_exp01_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (1, 0, 0): base_path_scrunchy + "results_exp02_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0, 1, 0): base_path_scrunchy + "results_exp03_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0.08, 1, 0): base_path_scrunchy + "results_exp04_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (1, 1, 0): base_path_scrunchy + "results_exp05_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0, 100, 0): base_path_scrunchy + "results_exp06_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0.08, 100, 0): base_path_scrunchy + "results_exp07_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (1, 100, 0): base_path_scrunchy + "results_exp08_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0, 1, 1): base_path_scrunchy + "results_exp09_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0.08, 1, 1): base_path_scrunchy + "results_exp10_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (1, 1, 1): base_path_scrunchy + "results_exp11_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0, 100, 1): base_path_scrunchy + "results_exp12_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (0.08, 100, 1): base_path_scrunchy + "results_exp13_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
        (1, 100, 1): base_path_scrunchy + "results_exp14_scrunchy_VAE_w100_100epochs/curvature_errors_stats.json",
    },
    "Sphere low dim": {
        (0, 0, 0): base_path + "curvatures_exp00_vae_s2_synthetic.pt",
        (0.08, 0, 0): base_path + "curvatures_exp02_vae_s2_synthetic.pt",
        (1, 0, 0): base_path + "curvatures_exp03_vae_s2_synthetic.pt",
        (0, 1, 0): base_path + "curvatures_exp04_vae_s2_synthetic.pt",
        (0.08, 1, 0): base_path + "curvatures_exp06_vae_s2_synthetic.pt",
        (1, 1, 0): base_path + "curvatures_exp07_vae_s2_synthetic.pt",
        (0, 100, 0): base_path + "curvatures_exp08_vae_s2_synthetic.pt",
        (0.08, 100, 0): base_path + "curvatures_exp10_vae_s2_synthetic.pt",
        (1, 100, 0): base_path + "curvatures_exp11_vae_s2_synthetic.pt",
        (0, 1, 1): base_path + "curvatures_exp12_vae_s2_synthetic.pt",
        (0.08, 1, 1): base_path + "curvatures_exp14_vae_s2_synthetic.pt",
        (1, 1, 1): base_path + "curvatures_exp15_vae_s2_synthetic.pt",
        (0, 100, 1): base_path + "curvatures_exp16_vae_s2_synthetic.pt",
        (0.08, 100, 1): base_path + "curvatures_exp18_vae_s2_synthetic.pt",
        (1, 100, 1): base_path + "curvatures_exp19_vae_s2_synthetic.pt",
        (0, 1, 2): base_path + "curvatures_exp20_vae_s2_synthetic.pt",
        (0.08, 1, 2): base_path + "curvatures_exp22_vae_s2_synthetic.pt",
        (1, 1, 2): base_path + "curvatures_exp23_vae_s2_synthetic.pt",
        (0, 100, 2): base_path + "curvatures_exp24_vae_s2_synthetic.pt",
        (0.08, 100, 2): base_path + "curvatures_exp26_vae_s2_synthetic.pt",
        (1, 100, 2): base_path + "curvatures_exp27_vae_s2_synthetic.pt",
    },
    "Sphere high dim": {
        (0, 0, 0): base_path + "curvatures_exp00_vae_sphere_high_dim.pt",
        (0.08, 0, 0): base_path + "curvatures_exp01_vae_sphere_high_dim.pt",
        (1, 0, 0): base_path + "curvatures_exp02_vae_sphere_high_dim.pt",
        (0, 1, 0): base_path + "curvatures_exp03_vae_sphere_high_dim.pt",
        (0.08, 1, 0): base_path + "curvatures_exp04_vae_sphere_high_dim.pt",
        (1, 1, 0): base_path + "curvatures_exp05_vae_sphere_high_dim.pt",
        (0, 100, 0): base_path + "curvatures_exp06_vae_sphere_high_dim.pt",
        (0.08, 100, 0): base_path + "curvatures_exp07_vae_sphere_high_dim.pt",
        (1, 100, 0): base_path + "curvatures_exp08_vae_sphere_high_dim.pt",
        (0, 1, 1): base_path + "curvatures_exp09_vae_sphere_high_dim.pt",
        (0.08, 1, 1): base_path + "curvatures_exp10_vae_sphere_high_dim.pt",
        (1, 1, 1): base_path + "curvatures_exp11_vae_sphere_high_dim.pt",
        (0, 100, 1): base_path + "curvatures_exp12_vae_sphere_high_dim.pt",
        (0.08, 100, 1): base_path + "curvatures_exp13_vae_sphere_high_dim.pt",
        (1, 100, 1): base_path + "curvatures_exp14_vae_sphere_high_dim.pt",
        (0, 1, 2): base_path + "curvatures_exp15_vae_sphere_high_dim.pt",
        (0.08, 1, 2): base_path + "curvatures_exp16_vae_sphere_high_dim.pt",
        (1, 1, 2): base_path + "curvatures_exp17_vae_sphere_high_dim.pt",
        (0, 100, 2): base_path + "curvatures_exp18_vae_sphere_high_dim.pt",
        (0.08, 100, 2): base_path + "curvatures_exp19_vae_sphere_high_dim.pt",
        (1, 100, 2): base_path + "curvatures_exp20_vae_sphere_high_dim.pt",
    },
}

datasets = [
    "Curve low dim", "Curve high dim",
    "Sphere low dim", "Sphere high dim",
    "Torus low dim", "Torus high dim"
]

generate_curv_table_from_json_lookup(
    file_lookup=file_lookup,
    datasets=datasets,
    save_path="tables/m_vae_full_table.tex"
)

cannonical order [(0, 0, 0), (0.08, 0, 0), (1, 0, 0), (0, 1, 0), (0.08, 1, 0), (1, 1, 0), (0, 100, 0), (0.08, 100, 0), (1, 100, 0), (0, 1, 1), (0.08, 1, 1), (1, 1, 1), (0, 100, 1), (0.08, 100, 1), (1, 100, 1), (0, 1, 2), (0.08, 1, 2), (1, 1, 2), (0, 100, 2), (0.08, 100, 2), (1, 100, 2)]
1.179777979850769 29.426446557044983
3.712449550628662 63.25041651725769
2.3447751998901367 33.83088707923889
4.565675735473633 66.79142713546753
1.0479098558425903 28.280743956565857
3.294971466064453 58.759331703186035
0.9021957516670227 26.488658785820007
3.512681484222412 62.16265559196472
1.6170865297317505 36.08163893222809
3.127671718597412 55.818575620651245
1.12660551071167 31.582608819007874
2.971247673034668 57.7905535697937
1.6086452007293701 37.076568603515625
3.1385881900787354 60.54585576057434
0.9011249542236328 26.798734068870544
2.7608325481414795 61.57147288322449
1.496587872505188 29.29341197013855
3.158857822418213 62.33876943588257
0.907791793346405 35.44946014881134
3.242635250091